In [54]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator


from langchain_huggingface import HuggingFaceEndpoint
from langchain_huggingface.chat_models import ChatHuggingFace
from dotenv import load_dotenv
import os

In [55]:
load_dotenv()

# Initialize HuggingFace LLM
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation",
    huggingfacehub_api_token=os.getenv("HUGGINGFACE_API_KEY")
)

model = ChatHuggingFace(llm=llm)

In [ ]:
# Define the state structure

class EvaluationSchema(BaseModel):

    feedback: str = Field(description='Detailed feedbackfor the essay')
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [ ]:
# Wrap the model with structured output

structured_model = model.with_structured_output(
    EvaluationSchema,
    method="json_schema"
)

In [59]:
essay = """🇧🇩 Role of Artificial Intelligence (AI) in Bangladesh

Artificial Intelligence (AI) is becoming one of the most transformative technologies in Bangladesh. Although the country is still in the early stages of adoption, AI is rapidly influencing major sectors—improving efficiency, enabling automation, and supporting data-driven decision-making.

Below is a structured breakdown:

✅ 1. Agriculture (Krishi)

Bangladesh is an agricultural country, and AI has huge potential here.

Current & Emerging Use Cases:

Crop disease detection using computer vision (e.g., rice blast, leaf blight).

AI-powered crop prediction, yield forecasting.

Smart irrigation and soil analysis.

Drones for land surveying and monitoring.

Impact:
Higher crop yield, reduced losses, better resource management.

✅ 2. Healthcare

AI is improving accessibility to medical services.

Use Cases:

Diagnostic support (X-ray, MRI, CT scan analysis).

AI chatbots in hospitals (e.g., appointment management).

Prediction models for dengue, COVID-19, and outbreaks.

Telemedicine with automated symptom analysis.

Impact:
Early diagnosis, reduced workload for doctors, improved rural healthcare accessibility.

✅ 3. Education

AI is transforming how students learn and how teachers teach.

Use Cases:

Personalized learning platforms.

AI-based English/Bangla writing assistants.

Cheating detection in online exams.

Automated grading systems.

Impact:
Better learning outcomes, reduced gaps between rural and urban education.

✅ 4. Financial Services (FinTech)

Bangladesh’s fintech ecosystem (bKash, Nagad, Rocket) is already using AI.

Use Cases:

Fraud detection for mobile banking.

Customer support chatbots.

Credit scoring using ML for microloans.

Transaction prediction & financial analytics.

Impact:
Safer digital transactions, more inclusive loan systems.

✅ 5. Government & Public Services (Smart Bangladesh Vision 2041)

The government is integrating AI into digital governance.

Use Cases:

National ID verification using face recognition.

Smart traffic management (pilot projects).

AI-based disaster prediction (cyclones, floods).

E-government automation in services.

Impact:
Faster public service delivery, improved disaster readiness.

✅ 6. Transportation & Smart Cities

Dhaka’s traffic problem is a major challenge—AI can help.

Use Cases:

Intelligent traffic signal control.

Vehicle detection and camera-based monitoring.

Predictive traffic congestion models.

Smart parking systems.

Impact:
Reduced congestion, safer roads.

✅ 7. Industry, RMG, and Manufacturing

Bangladesh’s RMG (Ready-Made Garments) is a major economic driver.

Use Cases:

Quality control using computer vision.

Predictive maintenance for machinery.

Automated pattern cutting and design.

Worker safety monitoring.

Impact:
Higher quality output, lower production costs.

✅ 8. Tourism & Service Industry

AI makes tourism more accessible.

Use Cases:

AI-powered travel assistance.

Virtual tour guides.

Sentiment analysis of traveler feedback.

Impact:
Better customer experience and digital tourism growth."""

In [ ]:
# Define the state structure

class State(TypedDict):
    essay: str

    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str

    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [61]:
def evaluate_language(state: State):
    prompt = (
        "Evaluate the language quality of the following essay. "
        "Return JSON with fields: feedback, score.\n\n"
        f"{state['essay']}"
    )

    output = structured_model.invoke(prompt)
    # output is dict → output["feedback"], output["score"]
    return {
        "language_feedback": output["feedback"],
        "individual_scores": [output["score"]]
    }


def evaluate_analysis(state: State):
    prompt = (
        "Evaluate the depth of analysis of the following essay. "
        "Return JSON with fields: feedback, score.\n\n"
        f"{state['essay']}"
    )

    output = structured_model.invoke(prompt)
    return {
        "analysis_feedback": output["feedback"],
        "individual_scores": [output["score"]]
    }


def evaluate_thought(state: State):
    prompt = (
        "Evaluate the clarity of thought of the following essay. "
        "Return JSON with fields: feedback, score.\n\n"
        f"{state['essay']}"
    )

    output = structured_model.invoke(prompt)
    return {
        "clarity_feedback": output["feedback"],
        "individual_scores": [output["score"]]
    }


def final_evaluation(state: State):
    prompt = f"""
Create a summarized and cohesive evaluation based on:

Language Feedback:
{state['language_feedback']}

Analysis Feedback:
{state['analysis_feedback']}

Clarity Feedback:
{state['clarity_feedback']}

Write a 3–5 sentence conclusion.
"""

    overall_feedback = model.invoke(prompt).content

    avg_score = sum(state["individual_scores"]) / len(state["individual_scores"])

    return {
        "overall_feedback": overall_feedback,
        "avg_score": avg_score
    }

In [ ]:
graph = StateGraph(State)

# Add nodes
graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_evaluation", final_evaluation)


# Define edges for parallel execution
graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_thought")

graph.add_edge("evaluate_language", "final_evaluation")
graph.add_edge("evaluate_analysis", "final_evaluation")
graph.add_edge("evaluate_thought", "final_evaluation")

graph.add_edge("final_evaluation", END)

# Compile the workflow
workflow = graph.compile()


In [63]:

initial_state = {
    "essay": essay,
    "individual_scores": []  # required for aggregated list
}

result = workflow.invoke(initial_state)
result

{'essay': '🇧🇩 Role of Artificial Intelligence (AI) in Bangladesh\n\nArtificial Intelligence (AI) is becoming one of the most transformative technologies in Bangladesh. Although the country is still in the early stages of adoption, AI is rapidly influencing major sectors—improving efficiency, enabling automation, and supporting data-driven decision-making.\n\nBelow is a structured breakdown:\n\n✅ 1. Agriculture (Krishi)\n\nBangladesh is an agricultural country, and AI has huge potential here.\n\nCurrent & Emerging Use Cases:\n\nCrop disease detection using computer vision (e.g., rice blast, leaf blight).\n\nAI-powered crop prediction, yield forecasting.\n\nSmart irrigation and soil analysis.\n\nDrones for land surveying and monitoring.\n\nImpact:\nHigher crop yield, reduced losses, better resource management.\n\n✅ 2. Healthcare\n\nAI is improving accessibility to medical services.\n\nUse Cases:\n\nDiagnostic support (X-ray, MRI, CT scan analysis).\n\nAI chatbots in hospitals (e.g., appo